
# NB00 — Project Setup & Environment Bootstrap
## Investor Intelligence Platform — FIIs Brasil 🇧🇷
### CRISP-DM Phase: Business Understanding

| | |
|---|---|
| **Autores** | Fabiana ⚡️ Campanari · Pedro Vyctor Almeida |
| **Instituição** | PUC-SP · FACEI |
| **Professor** | Eduardo Savino Gomes |
| **Fase CRISP-DM** | Business Understanding |
| **Função** | Bootstrap executado UMA vez antes de NB01–NB07 |
| **Reprodutibilidade** | `RANDOM_SEED = 42` |

## 🎯 O que este notebook faz

1. Valida versões de Python, Java e PySpark (avisa, nunca quebra o kernel).  
2. Instala dependências ausentes.  
3. Cria estrutura de diretórios Medallion: `data/bronze` · `data/silver` · `data/gold` · `logs/` · `config/` · `src/`.  
4. Gera **`config/settings.py`** — fonte única de verdade de constantes (21 fontes, caminhos, seeds, Spark).  
5. Gera **`src/utils/logger.py`** — `get_logger()` + funções estruturadas para toda a pipeline.  
6. Gera stubs de API FastAPI, dashboard Streamlit e chatbot Groq.  
7. Executa validação final.

> ⚠️ **Execute este notebook ANTES de NB01–NB07.**
> Sem ele `config/settings.py` e `src/utils/logger.py` não existem.

## 🏛️ Arquitetura Medallion

```
21 FONTES MONITORADAS
      ↓
NB01 — BRONZE  →  data/external/  (dados brutos, congelados)
      ↓
NB02 — SILVER  →  data/silver/    (limpeza, normalização)
      ↓
NB03–NB07 — GOLD  →  data/gold/   (NLP, ranking, sentimento, dashboard)
```

## 📋 Fontes Monitoradas — 21 no Total

| # | Portal/Fonte | Tipo | Método | URL base |
|---|---|---|---|---|
| 1 | InfoMoney | Editorial | RSS | infomoney.com.br/feed/ |
| 2 | Empiricus | Editorial | RSS | empiricus.com.br/feed/ |
| 3 | Money Times | Editorial | RSS | moneytimes.com.br/feed/ |
| 4 | Seu Dinheiro | Editorial | RSS | seudinheiro.com/feed/ |
| 5 | Exame Invest | Editorial | RSS | exame.com/feed/ |
| 6 | CNN Brasil Business | Editorial | RSS | cnnbrasil.com.br/feed/ |
| 7 | Suno Research | Editorial | RSS supl. | sunoresearch.com.br/feed/ |
| 8 | E-Investidor Estadão | Editorial | RSS supl. | einvestidor.estadao.com.br/feed |
| 9 | NeoFeed | Editorial | RSS supl. | neofeed.com.br/feed/ |
| 10 | Toro Investimentos | Editorial | RSS supl. + fallback scraping | blog.toroinvestimentos.com.br/feed/ |
| 11 | Funds Explorer | Editorial | Scraping | fundsexplorer.com.br/ranking |
| 12 | Status Invest | Editorial | Scraping | statusinvest.com.br/fundos-imobiliarios |
| 13 | Clube FII | Editorial | Scraping | clubefii.com.br |
| 14 | FIIs.com.br | Editorial | Scraping | fiis.com.br |
| 15 | Portal do FII | Editorial | Scraping + fallback RSS | portaldofii.com.br |
| 16 | Investidor10 | Editorial | Scraping | investidor10.com.br/fiis/ |
| 17 | Eu Quero Investir | Editorial | Scraping | euqueroinvestir.com/fundos-imobiliarios/ |
| 18 | Bora Investir (B3) | Editorial | Scraping | borainvestir.b3.com.br |
| 19 | XP Conteúdos | Editorial | Scraping | conteudos.xpi.com.br |
| 20 | Investing Brasil | Editorial | Scraping | br.investing.com/news/stock-market-news |
| **21** | **Reddit / Google News (fallback)** | **Social / Behavioral** | **PRAW (quando disponível) + Google News RSS (fallback)** | r/investimentos · r/farialimabets · news.google.com |


## 📦 Seção 1 — Imports e Diagnóstico do Ambiente

Importa módulos do sistema e imprime informações básicas: versão Python, SO e raiz do projeto.

In [1]:

import sys
import os
import platform
import subprocess
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

CURRENT_DIR  = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

print("✅ Imports loaded")
print(f"   Project root : {PROJECT_ROOT}")
print(f"   Python       : {sys.version.split()[0]}")
print(f"   Platform     : {platform.platform()}")

if sys.version_info < (3, 10):
    print(f"   ⚠️  Python {sys.version.split()[0]} — recomendado 3.10+")
else:
    print(f"   ✅ Python {sys.version.split()[0]} — OK")

✅ Imports loaded
   Project root : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform
   Python       : 3.12.12
   Platform     : macOS-26.5.1-arm64-arm-64bit
   ✅ Python 3.12.12 — OK



## 📥 Seção 2 — Instalação de Dependências

Garante que todas as bibliotecas estejam disponíveis no kernel atual.
Pode ser re-executada sem efeitos colaterais.

In [2]:
import subprocess, sys

PACKAGES = ['pyspark', 'pandas', 'numpy', 'pyarrow', 'rank-bm25', 'textblob', 'nltk', 'scikit-learn', 'plotly', 'streamlit', 'fastapi', 'feedparser', 'beautifulsoup4', 'groq', 'requests', 'python-dateutil', 'uvicorn']

print('📥 Verificando e instalando dependências...')
missing = []
for pkg in PACKAGES:
    mod = pkg.replace('-', '_').split('==')[0]
    try:
        __import__(mod)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f'   Instalando: {missing}')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet'] + missing
    )
    print('   ✅ Instalação concluída')
else:
    print('   ✅ Todas as dependências já presentes')

📥 Verificando e instalando dependências...


   Instalando: ['scikit-learn', 'beautifulsoup4', 'python-dateutil']
   ✅ Instalação concluída



## ☕ Seção 3 — Verificação de Java e PySpark

Confirma Java disponível e executa teste funcional com PySpark.
PySpark depende de Java — detectar cedo evita falhas silenciosas.

In [3]:

# ── Java check ────────────────────────────────────────────────────────────────
print("☕ Java (required for PySpark)")
try:
    result = subprocess.run(
        ['java', '-version'], capture_output=True, timeout=10, check=False
    )
    raw  = (result.stderr or result.stdout or b'')
    out  = raw.decode(errors='replace').strip()
    line = out.splitlines()[0] if out.splitlines() else 'version unknown'
    print(f"   ✅ {line}")
except FileNotFoundError:
    print("   ❌ Java não encontrado — instale JDK 11+")
    print("      Linux : sudo apt install openjdk-11-jdk")
    print("      Mac   : brew install openjdk@11")
    print("      Win   : https://adoptium.net/")
except Exception as e:
    print(f"   ⚠️  Java check falhou: {e}")

# ── Dependency check ──────────────────────────────────────────────────────────
print("\n📦 Dependency Check")
REQUIRED = [
    ('pyspark',    'PySpark',        'pyspark'),
    ('pandas',     'Pandas',         'pandas'),
    ('numpy',      'NumPy',          'numpy'),
    ('pyarrow',    'PyArrow',        'pyarrow'),
    ('rank_bm25',  'BM25',           'rank-bm25'),
    ('textblob',   'TextBlob',       'textblob'),
    ('nltk',       'NLTK',           'nltk'),
    ('sklearn',    'Scikit-learn',   'scikit-learn'),
    ('plotly',     'Plotly',         'plotly'),
    ('streamlit',  'Streamlit',      'streamlit'),
    ('fastapi',    'FastAPI',        'fastapi'),
    ('feedparser', 'Feedparser',     'feedparser'),
    ('bs4',        'BeautifulSoup4', 'beautifulsoup4'),
    ('groq',       'Groq',           'groq'),
    ('requests',   'Requests',       'requests'),
]
missing_packages = []
for mod, name, pip_name in REQUIRED:
    try:
        __import__(mod)
        print(f"   ✅ {name}")
    except ImportError:
        print(f"   ❌ {name}")
        missing_packages.append(pip_name)

if missing_packages:
    print(f"\n   ❌ Ausentes: pip install {' '.join(missing_packages)}")
else:
    print("\n   ✅ Todas as dependências presentes")

☕ Java (required for PySpark)
   ✅ openjdk version "17.0.19" 2026-04-21

📦 Dependency Check
   ✅ PySpark
   ✅ Pandas
   ✅ NumPy
   ✅ PyArrow
   ✅ BM25
   ✅ TextBlob
   ✅ NLTK
   ✅ Scikit-learn
   ✅ Plotly
   ✅ Streamlit
   ✅ FastAPI
   ✅ Feedparser
   ✅ BeautifulSoup4
   ✅ Groq
   ✅ Requests

   ✅ Todas as dependências presentes


In [4]:

# ── PySpark functional test ───────────────────────────────────────────────────
spark_available = False
try:
    import os as _os, sys as _sys
    _os.environ["PYSPARK_PYTHON"] = _sys.executable
    _os.environ["PYSPARK_DRIVER_PYTHON"] = _sys.executable
    from pyspark.sql import SparkSession
    import pyspark

    _spark = (
        SparkSession.builder
        .appName("NB00_validation")
        .master("local[2]")
        .config("spark.driver.memory", "1g")
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )
    _spark.sparkContext.setLogLevel("ERROR")

    # Quick smoke test
    _df = _spark.createDataFrame([(1, "fii"), (2, "dividendo")], ["id", "term"])
    assert _df.count() == 2
    _spark.stop()

    spark_available = True
    print(f"✅ PySpark {pyspark.__version__} — smoke test passed")

except Exception as e:
    print(f"⚠️  PySpark não disponível: {e}")
    print("   NB01–NB07 requerem Java + PySpark")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/14 15:29:27 WARN Utils: Your hostname, MacBook-Air-100.local, resolves to a loopback address: 127.0.0.1; using 192.168.15.4 instead (on interface en0)
26/06/14 15:29:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 15:29:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ PySpark 4.1.2 — smoke test passed



## 🗂️ Seção 4 — Estrutura de Diretórios

Cria a estrutura Medallion completa:
`data/external` · `data/bronze` · `data/silver` · `data/gold` · `logs/` · `config/` · `src/utils/`.

In [5]:

DIRS = [
    "data/external",
    "data/bronze",
    "data/silver",
    "data/gold/word_count",
    "data/gold/tfidf_bm25",
    "data/gold/sentiment",
    "data/gold/marketing_intelligence",
    "data/gold/dashboard",
    "logs",
    "config",
    "src/utils",
    "src/api",
    "src/pipeline",
    "src/nlp",
    "docs/governance",
    "docs/architecture",
    "docs/methodology",
    "notebooks",
    "presentations",
    "api",
    "dashboard/chatbot",
    "dashboard/pages",
    "output",
    "assets",
]

for d in DIRS:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

# Ensure __init__.py files
for pkg in ["config", "src", "src/utils", "src/api", "src/pipeline", "src/nlp"]:
    init = PROJECT_ROOT / pkg / "__init__.py"
    if not init.exists():
        init.write_text("")

print("✅ Estrutura de diretórios criada")
for d in DIRS[:10]:
    print(f"   📁 {d}/")
print(f"   ... +{len(DIRS)-10} diretórios adicionais")

✅ Estrutura de diretórios criada
   📁 data/external/
   📁 data/bronze/
   📁 data/silver/
   📁 data/gold/word_count/
   📁 data/gold/tfidf_bm25/
   📁 data/gold/sentiment/
   📁 data/gold/marketing_intelligence/
   📁 data/gold/dashboard/
   📁 logs/
   📁 config/
   ... +14 diretórios adicionais



## ⚙️ Seção 5 — Geração de `config/settings.py`

Fonte única de verdade para todo 0 pipeline.  
Contém: caminhos Medallion, 21 fontes, constantes Spark, parâmetros HTTP, filtros FII e configuração Reddit.

In [6]:
SETTINGS_PY = """
\"\"\"
config/settings.py — Single Source of Truth
Investor Intelligence Platform — FIIs Brasil
Gerado por NB00. NÃO edite manualmente — re-execute NB00 para regenerar.
\"\"\"
from pathlib import Path
import os

# ─── Reprodutibilidade ────────────────────────────────────────────────────────
RANDOM_SEED = 42

# ─── Raiz do projeto ──────────────────────────────────────────────────────────
PROJECT_ROOT = Path(__file__).resolve().parent.parent

# ─── Caminhos Medallion ───────────────────────────────────────────────────────
DATA_DIR     = PROJECT_ROOT / "data"
EXTERNAL_DIR = DATA_DIR / "external"       # Bronze (frozen raw data)
BRONZE_DIR   = DATA_DIR / "bronze"         # Bronze (processed)
SILVER_DIR   = DATA_DIR / "silver"         # Silver (clean)
GOLD_DIR     = DATA_DIR / "gold"           # Gold (analytics)
LOGS_DIR     = PROJECT_ROOT / "logs"
CONFIG_DIR   = PROJECT_ROOT / "config"
SRC_DIR      = PROJECT_ROOT / "src"

# ─── Spark ────────────────────────────────────────────────────────────────────
SPARK_APP_NAME      = "FIIIntelligencePlatform"
SPARK_DRIVER_MEMORY = "4g"
SPARK_SHUFFLE_PARTS = "4"

# ─── HTTP client ──────────────────────────────────────────────────────────────
REQUEST_TIMEOUT  = 20      # seconds
MAX_RETRIES      = 3
RETRY_BACKOFF    = 2       # exponential base (2^attempt seconds)
RATE_LIMIT_DELAY = 1.0     # seconds between requests

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/115.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/118.0",
]

# ─── RSS Feeds primários (6 feeds — settings.py) ─────────────────────────────
# NB01 adiciona outros 4 SUPPLEMENTAL_RSS_FEEDS + 10 PORTAL_TARGETS = 21 total
RSS_FEEDS = [
    "https://www.infomoney.com.br/feed/",          # 1. InfoMoney
    "https://empiricus.com.br/feed/",              # 2. Empiricus
    "https://www.moneytimes.com.br/feed/",         # 3. Money Times
    "https://www.seudinheiro.com/feed/",           # 4. Seu Dinheiro
    "https://exame.com/feed/",                     # 5. Exame Invest
    "https://www.cnnbrasil.com.br/feed/",          # 6. CNN Brasil Business
]

# ─── FII Filter Terms ─────────────────────────────────────────────────────────
FII_FILTER_TERMS = [
    "fii", "fundo imobiliário", "fundo imobiliario",
    "fiis", "fundos imobiliários", "fundos imobiliarios",
    "reit", "dividendo", "dividendos", "provento", "proventos",
    "vacância", "vacancia", "cotista", "cotistas",
    "yield", "p/vp", "tijolo", "papel", "logística", "logistica",
    "shopping", "laje corporativa", "galpão", "galpao",
    "fundo de investimento imobiliário",
    "fundo de investimento imobiliario",
]

# ─── Reddit ───────────────────────────────────────────────────────────────────
REDDIT_CLIENT_ID     = os.getenv("REDDIT_CLIENT_ID", "")
REDDIT_CLIENT_SECRET = os.getenv("REDDIT_CLIENT_SECRET", "")
REDDIT_USER_AGENT    = os.getenv(
    "REDDIT_USER_AGENT",
    "FIIIntelligencePlatform/1.0 (academic; PUC-SP FACEI)",
)
REDDIT_SUBREDDITS    = ["investimentos", "farialimabets"]
REDDIT_API_AVAILABLE = bool(REDDIT_CLIENT_ID and REDDIT_CLIENT_SECRET)
"""

settings_path = PROJECT_ROOT / "config" / "settings.py"
settings_path.write_text(SETTINGS_PY.lstrip())
print(f"✅ config/settings.py gerado")
print(f"   Caminho: {settings_path}")

✅ config/settings.py gerado
   Caminho: /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/config/settings.py



## 📝 Seção 6 — Geração de `src/utils/logger.py`

Logger estruturado com handlers de console e arquivo.  
Expõe funções tipadas para cada evento de pipeline:  
`log_source_success` · `log_source_failure` · `log_retry` · `log_timeout` · `log_duplicate` · `log_dataset_frozen` · `log_quality_check` · **`log_spark_start(logger, app_name, spark_version)`** (3 argumentos).

In [7]:
LOGGER_PY = """
\"\"\"src/utils/logger.py — Structured Pipeline Logger
Investor Intelligence Platform — FIIs Brasil
Gerado por NB00.
\"\"\"
import logging
import sys
from pathlib import Path
from datetime import datetime, timezone

_LOG_DIR = Path(__file__).resolve().parent.parent.parent / "logs"
_LOG_DIR.mkdir(parents=True, exist_ok=True)


def get_logger(name: str) -> logging.Logger:
    \"\"\"Factory: devolve logger com handlers de console + arquivo.\"\"\"
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger

    logger.setLevel(logging.DEBUG)
    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%dT%H:%M:%S",
    )

    # Console — INFO+
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    # Arquivo — DEBUG+ (rotação diária por nome)
    _date = datetime.now(timezone.utc).strftime("%Y%m%d")
    log_file = _LOG_DIR / f"{name.replace('.', '_')}_{_date}.log"
    fh = logging.FileHandler(log_file, encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    logger.propagate = False
    return logger


def ingestion_logger() -> logging.Logger:
    \"\"\"Logger pré-configurado para toda a pipeline de ingestão.\"\"\"
    return get_logger("fii_pipeline.ingestion")


# ─── Event functions ──────────────────────────────────────────────────────────

def log_source_success(logger: logging.Logger, source: str, count: int) -> None:
    logger.info(f"SOURCE_OK   | {source} | {count} records")


def log_source_failure(logger: logging.Logger, source: str, error: str) -> None:
    logger.warning(f"SOURCE_FAIL | {source} | {error}")


def log_retry(
    logger: logging.Logger, source: str, attempt: int, max_retries: int
) -> None:
    logger.debug(f"RETRY       | {source} | attempt {attempt}/{max_retries}")


def log_timeout(logger: logging.Logger, source: str, timeout: float) -> None:
    logger.warning(f"TIMEOUT     | {source} | exceeded {timeout}s")


def log_duplicate(logger: logging.Logger, text: str) -> None:
    logger.debug(f"DUPLICATE   | {text[:80]}")


def log_dataset_frozen(logger: logging.Logger, path: str, count: int) -> None:
    logger.info(
        f"DATASET_FROZEN | {path} | {count} records | downstream reads only"
    )


def log_quality_check(logger: logging.Logger, **kwargs) -> None:
    kv = " | ".join(f"{k}={v}" for k, v in kwargs.items())
    logger.info(f"QUALITY_CHECK | {kv}")


def log_spark_start(
    logger: logging.Logger, app_name: str, spark_version: str
) -> None:
    \"\"\"Log Spark session start. Requires 3 positional args (logger, app_name, spark_version).\"\"\"
    logger.info(f"SPARK_START | app={app_name} | spark={spark_version}")
"""

logger_path = PROJECT_ROOT / "src" / "utils" / "logger.py"
logger_path.write_text(LOGGER_PY.lstrip())
print(f"✅ src/utils/logger.py gerado")
print(f"   Funções exportadas:")
for fn in ["get_logger", "ingestion_logger", "log_source_success",
           "log_source_failure", "log_retry", "log_timeout",
           "log_duplicate", "log_dataset_frozen", "log_quality_check",
           "log_spark_start"]:
    print(f"     • {fn}()")

✅ src/utils/logger.py gerado
   Funções exportadas:
     • get_logger()
     • ingestion_logger()
     • log_source_success()
     • log_source_failure()
     • log_retry()
     • log_timeout()
     • log_duplicate()
     • log_dataset_frozen()
     • log_quality_check()
     • log_spark_start()



## 🔌 Seção 7 — Stubs de API FastAPI + Streamlit + Groq

Cria esqueletos prontos para a Fase 2 (productização).

In [8]:

# ── FastAPI stub ──────────────────────────────────────────────────────────────
FASTAPI_PY = """
\"\"\"api/app.py — FastAPI skeleton (Phase 2). Generated by NB00.\"\"\"
from fastapi import FastAPI
from pathlib import Path
import pandas as pd, json

app = FastAPI(
    title="FII Intelligence API",
    description="Investor Intelligence Platform — FIIs Brasil",
    version="1.0.0",
)

GOLD = Path(__file__).parent.parent / "data" / "gold"

@app.get("/health")
def health():
    return {"status": "ok", "version": "1.0.0"}

@app.get("/articles")
def articles(limit: int = 20):
    p = GOLD / "dashboard" / "dashboard_articles.parquet"
    if not p.exists():
        return {"error": "Run NB07 first"}
    df = pd.read_parquet(p).head(limit)
    return df.to_dict(orient="records")

@app.get("/sources")
def sources():
    p = GOLD / "dashboard" / "dashboard_sources.parquet"
    if not p.exists():
        return {"error": "Run NB07 first"}
    df = pd.read_parquet(p)
    return df.to_dict(orient="records")
"""

(PROJECT_ROOT / "api" / "app.py").write_text(FASTAPI_PY.lstrip())
print("✅ api/app.py gerado")

# ── Streamlit stub ────────────────────────────────────────────────────────────
STREAMLIT_HOME = """
\"\"\"dashboard/Home.py — Streamlit home (Phase 2). Generated by NB00.\"\"\"
import streamlit as st
from pathlib import Path
import pandas as pd

st.set_page_config(page_title="FII Intelligence", page_icon="🏢", layout="wide")

st.title("🏢 Investor Intelligence Platform — FIIs Brasil")
st.markdown("### Plataforma de Inteligência de Marketing para FIIs")

GOLD = Path(__file__).parent.parent / "data" / "gold" / "dashboard"

if (GOLD / "dashboard_articles.parquet").exists():
    df = pd.read_parquet(GOLD / "dashboard_articles.parquet")
    st.metric("Artigos monitorados", len(df))
    st.dataframe(df.head(20), use_container_width=True)
else:
    st.warning("⚠️ Execute NB01–NB07 para gerar os datasets do dashboard.")
"""
(PROJECT_ROOT / "dashboard" / "Home.py").write_text(STREAMLIT_HOME.lstrip())
print("✅ dashboard/Home.py gerado")

# ── Groq chatbot stub ─────────────────────────────────────────────────────────
GROQ_CLIENT = """
\"\"\"dashboard/chatbot/groq_client.py — Groq chatbot stub. Generated by NB00.\"\"\"
import os
from groq import Groq

DISCLAIMER = (
    \"Esta análise é gerada por IA com base em dados públicos. \"
    \"Não constitui recomendação de investimento. \"
    \"Consulte um assessor financeiro certificado antes de tomar decisões.\"
)

def get_groq_client() -> Groq:
    api_key = os.getenv("GROQ_API_KEY", "")
    if not api_key:
        raise ValueError("GROQ_API_KEY não definido no ambiente.")
    return Groq(api_key=api_key)

def chat(user_message: str, context: str = "") -> str:
    client = get_groq_client()
    system_prompt = (
        "Você é um assistente especializado em fundos imobiliários brasileiros (FIIs). "
        "Responda sempre em português. "
        f"Contexto de mercado: {context[:2000] if context else 'Não disponível.'}\\n\\n"
        f"AVISO LEGAL: {DISCLAIMER}"
    )
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        max_tokens=800,
        temperature=0.3,
    )
    return response.choices[0].message.content + f"\\n\\n---\\n⚠️ {DISCLAIMER}"
"""
(PROJECT_ROOT / "dashboard" / "chatbot" / "groq_client.py").write_text(GROQ_CLIENT.lstrip())
print("✅ dashboard/chatbot/groq_client.py gerado")

✅ api/app.py gerado
✅ dashboard/Home.py gerado
✅ dashboard/chatbot/groq_client.py gerado



## 🔐 Seção 8 — Arquivos de Configuração do Projeto

Gera `.env.example`, `.gitignore`, `requirements.txt` e `Makefile`.

In [9]:

# .env.example
(PROJECT_ROOT / ".env.example").write_text(
    "# Reddit (optional — required for full PRAW ingestion)\n"
    "REDDIT_CLIENT_ID=your_client_id_here\n"
    "REDDIT_CLIENT_SECRET=your_client_secret_here\n"
    "REDDIT_USER_AGENT=FIIIntelligencePlatform/1.0 (academic; PUC-SP)\n\n"
    "# Groq (optional — required for chatbot in Phase 2)\n"
    "GROQ_API_KEY=your_groq_api_key_here\n\n"
    "# Reproducibility\n"
    "PYTHONHASHSEED=42\n"
)
print("✅ .env.example")

# .gitignore
GITIGNORE = """
# Python
__pycache__/
*.py[cod]
*.egg-info/
.eggs/
dist/
build/
*.so
.venv/
venv/
env/

# Jupyter
.ipynb_checkpoints/

# Data (large files — use DVC or LFS if needed)
data/external/*.parquet
data/silver/*.parquet
data/gold/**/*.parquet

# Logs
logs/

# Secrets
.env
*.env

# OS
.DS_Store
Thumbs.db
"""
(PROJECT_ROOT / ".gitignore").write_text(GITIGNORE.lstrip())
print("✅ .gitignore")

# requirements.txt
REQUIREMENTS = """
pyspark>=3.4
pandas>=2.0
numpy>=1.24
pyarrow>=12.0
rank-bm25>=0.2.2
textblob>=0.17
nltk>=3.8
scikit-learn>=1.3
plotly>=5.17
streamlit>=1.28
fastapi>=0.103
uvicorn>=0.23
feedparser>=6.0
beautifulsoup4>=4.12
groq>=0.4
requests>=2.31
python-dateutil>=2.8
"""
(PROJECT_ROOT / "requirements.txt").write_text(REQUIREMENTS.lstrip())
print("✅ requirements.txt")

# Makefile
MAKEFILE = """
.PHONY: setup ingest pipeline clean

setup:
\tpip install -r requirements.txt

ingest:
\tjupyter nbconvert --to notebook --execute notebooks/NB00_setup.ipynb
\tjupyter nbconvert --to notebook --execute notebooks/NB01_bronze_ingestion.ipynb

pipeline:
\tfor nb in 02 03 04 05 06 07; do \\
\t\tjupyter nbconvert --to notebook --execute notebooks/NB$${nb}_*.ipynb; \\
\tdone

clean:
\trm -rf data/external data/silver data/gold logs
\tmkdir -p data/external data/silver data/gold logs
"""
(PROJECT_ROOT / "Makefile").write_text(MAKEFILE.lstrip())
print("✅ Makefile")

✅ .env.example
✅ .gitignore
✅ requirements.txt
✅ Makefile



## 📋 Seção 9 — Documentação de Governança e CRISP-DM

Gera `docs/BM25_FOUNDATION.md`, `docs/architecture/bronze_schema.md` e outros artefatos de governança.

In [10]:

import textwrap

DOCS = {
    "docs/BM25_FOUNDATION.md": """
# BM25 Foundation — Mathematical Reference
**Investor Intelligence Platform – FIIs Brasil**

## Formula
```
BM25(D,Q) = Σ IDF(qi) · [f(qi,D)·(k1+1)] / [f(qi,D) + k1·(1−b+b·|D|/avgdl)]
```
| Parameter | Default | Effect |
|-----------|---------|--------|
| k1 | 1.5 | Term saturation — diminishing returns for repeated terms |
| b  | 0.75 | Length normalization — penalizes very long documents |

## Role in this project
- **BM25** = source relevance ranking in NB04
- **TF-IDF** = term weighting for topic modeling in NB04
- **Combined** = 0.4×TF-IDF + 0.6×BM25 after MinMax normalization

## XAI Alignment
Every BM25 score decomposes into per-term contributions. Fully auditable.

## Reference
Robertson & Zaragoza (2009). The Probabilistic Relevance Framework: BM25 and Beyond.
Foundations and Trends in IR, 3(4), 333–389.
""",
    "docs/architecture/bronze_schema.md": """
# Bronze Layer Schema Contract — 17 Fields
**Investor Intelligence Platform – FIIs Brasil**

| Field | Type | Nullable | Description |
|-------|------|----------|-------------|
| article_id | STRING | No | SHA-256(url) — deterministic PK |
| source | STRING | No | Portal domain |
| source_type | STRING | No | rss / scraping / reddit |
| title | STRING | No | Raw headline |
| content | STRING | Yes | Full body text |
| summary | STRING | Yes | Lead paragraph |
| url | STRING | No | Canonical URL |
| author | STRING | Yes | Author name (editorial metadata) |
| published_at | STRING | Yes | Raw date string (None for scraped content without date) |
| collected_at | STRING | No | ISO-8601 UTC collection timestamp |
| language | STRING | No | pt-br |
| tags | STRING | Yes | Comma-separated tags |
| query_used | STRING | Yes | FII filter term matched |
| ingestion_method | STRING | No | feedparser / requests+bs4 / praw |
| raw_html | STRING | Yes | Raw HTML element (scraping only) |
| content_hash | STRING | No | MD5(title + content[:500]) |
| metadata_json | STRING | Yes | Source-specific extras (JSON string) |

## Key Rules
- `published_at = None` for scraping content that has no real publication date
- `article_id = SHA-256(url)` — primary deduplication key
- `content_hash = MD5(title + content[:500])` — near-duplicate detection
""",
    "docs/governance/LGPD_ALIGNMENT.md": """
# LGPD Alignment
**Investor Intelligence Platform – FIIs Brasil**

## Data minimization
Only publicly available editorial and community content is collected.
No personal data profiling. Author names are editorial metadata from public articles.

## Processing basis
Legitimate interest (Art. 7, VI, LGPD): public financial intelligence.

## Data subjects
No individual profiling. Reddit usernames stored only as editorial metadata.

## Retention
Raw data retained for reproducibility only. No indefinite storage.

## Contact
Academic project — PUC-SP FACEI.
""",
    "docs/governance/RESPONSIBLE_AI.md": """
# Responsible AI Documentation
**Investor Intelligence Platform – FIIs Brasil**

## Explainability (XAI)
Every ranking signal (TF-IDF, BM25) decomposes into auditable per-term contributions.
Sentiment scores trace to specific lexicon entries.

## Transparency
Source origin preserved through the pipeline via `source`, `source_type`, `ingestion_method`.

## Scope boundary
This platform is analytical, not advisory. All chatbot outputs include a legal disclaimer.

## Bias awareness
Sentiment lexicon tuned for PT-BR financial domain. Generic tools (TextBlob VADER)
produce incorrect results in FII context (NB05 documents why).

## CRISP-DM alignment
Methodology documented end-to-end from Business Understanding through Deployment.
""",
}

for rel_path, content in DOCS.items():
    target = PROJECT_ROOT / rel_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(textwrap.dedent(content).lstrip())
    print(f"✅ {rel_path}")

✅ docs/BM25_FOUNDATION.md
✅ docs/architecture/bronze_schema.md
✅ docs/governance/LGPD_ALIGNMENT.md
✅ docs/governance/RESPONSIBLE_AI.md



## 🔁 Seção 10 — Reprodutibilidade e Seed Global

In [11]:

import random
import numpy as np
import os

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

print(f"✅ RANDOM_SEED = {RANDOM_SEED} aplicado")
print("   Python random : ✅")
print("   NumPy random  : ✅")
print("   PYTHONHASHSEED: set (efeito total se definido antes do kernel)")
print()
print("📌 Escopo de reprodutibilidade:")
print("   ✅ Frozen dataset (data/external/) → corpus idêntico")
print("   ✅ RANDOM_SEED=42 → resultados determinísticos")
print("   ✅ BM25 (stateless) → scores idênticos")
print("   ⚠️  Spark row ORDER → use .orderBy() explícito nos notebooks Gold")
print("   ⚠️  PYTHONHASHSEED → efeito pleno apenas se definido pré-kernel")

✅ RANDOM_SEED = 42 aplicado
   Python random : ✅
   NumPy random  : ✅
   PYTHONHASHSEED: set (efeito total se definido antes do kernel)

📌 Escopo de reprodutibilidade:
   ✅ Frozen dataset (data/external/) → corpus idêntico
   ✅ RANDOM_SEED=42 → resultados determinísticos
   ✅ BM25 (stateless) → scores idênticos
   ⚠️  Spark row ORDER → use .orderBy() explícito nos notebooks Gold
   ⚠️  PYTHONHASHSEED → efeito pleno apenas se definido pré-kernel



## ✅ Seção 11 — Validação Final

Verifica todos os artefatos gerados antes de liberar NB01.

In [12]:
ast = __import__("ast")

def _try_import(module: str) -> bool:
    s = str(PROJECT_ROOT)
    if s not in sys.path:
        sys.path.insert(0, s)
    try:
        __import__(module)
        return True
    except Exception:
        return False



def _syntax_ok(path) -> bool:
    p = Path(path)
    if not p.exists():
        return False
    try:
        ast.parse(p.read_text())
        return True
    except SyntaxError:
        return False


CHECKS = [
    ("Python 3.10+",                 sys.version_info >= (3, 10)),
    ("Spark (warn-only)",            spark_available),
    ("No missing packages",          len(missing_packages) == 0),
    ("data/external/ exists",        (PROJECT_ROOT / "data" / "external").exists()),
    ("config/settings.py",           (PROJECT_ROOT / "config" / "settings.py").exists()),
    ("src/utils/logger.py",          (PROJECT_ROOT / "src" / "utils" / "logger.py").exists()),
    ("api/app.py",                   (PROJECT_ROOT / "api" / "app.py").exists()),
    ("dashboard/Home.py",            (PROJECT_ROOT / "dashboard" / "Home.py").exists()),
    ("groq_client.py",               (PROJECT_ROOT / "dashboard" / "chatbot" / "groq_client.py").exists()),
    ("settings.py syntax OK",        _syntax_ok(PROJECT_ROOT / "config" / "settings.py")),
    ("logger.py syntax OK",          _syntax_ok(PROJECT_ROOT / "src" / "utils" / "logger.py")),
    ("api/app.py syntax OK",         _syntax_ok(PROJECT_ROOT / "api" / "app.py")),
    ("config.settings importable",   _try_import("config.settings")),
    ("src.utils.logger importable",  _try_import("src.utils.logger")),
    ("docs/BM25_FOUNDATION.md",      (PROJECT_ROOT / "docs" / "BM25_FOUNDATION.md").exists()),
    ("docs/bronze_schema.md",        (PROJECT_ROOT / "docs" / "architecture" / "bronze_schema.md").exists()),
    ("LGPD_ALIGNMENT.md",            (PROJECT_ROOT / "docs" / "governance" / "LGPD_ALIGNMENT.md").exists()),
    ("RESPONSIBLE_AI.md",            (PROJECT_ROOT / "docs" / "governance" / "RESPONSIBLE_AI.md").exists()),
    ("RANDOM_SEED = 42",             RANDOM_SEED == 42),
]

WARN_ONLY = {"Spark (warn-only)"}

print("=" * 65)
print("✅  FINAL VALIDATION — Investor Intelligence Platform 🇧🇷")
print("=" * 65)

all_ok = True
for name, ok in CHECKS:
    if not ok and name in WARN_ONLY:
        print(f"  ⚠️  {name} (non-blocking — JDK required for NB01+)")
    elif ok:
        print(f"  ✅ {name}")
    else:
        print(f"  ❌ {name}")
        all_ok = False

ACADEMIC = [
    "21 Monitored Sources (20 Editorial + Reddit)",
    "PySpark · Distributed Computing · RDD MapReduce",
    "BM25 (rank-bm25) · TF-IDF vectorization",
    "Contextual Sentiment — PT-BR Financial Lexicon (NB05)",
    "Medallion Architecture: Bronze → Silver → Gold",
    "FastAPI REST · Streamlit Dashboard · Groq Chatbot (stubs)",
    "Responsible AI · LGPD · EU AI Act alignment",
    "CRISP-DM methodology (NB00–NB07 lifecycle)",
]
print("\n📋 Academic Requirements:")
for item in ACADEMIC:
    print(f"  ✅ {item}")

print("\n" + "=" * 65)
if all_ok:
    print("🎉 NB00 COMPLETO — PRONTO PARA NB01")
else:
    print("❌ Corrija os itens acima antes de NB01")
print("   Próximo: 01_data_ingestion_bronze.ipynb")
print("=" * 65)

✅  FINAL VALIDATION — Investor Intelligence Platform 🇧🇷
  ✅ Python 3.10+
  ✅ Spark (warn-only)
  ✅ No missing packages
  ✅ data/external/ exists
  ✅ config/settings.py
  ✅ src/utils/logger.py
  ✅ api/app.py
  ✅ dashboard/Home.py
  ✅ groq_client.py
  ✅ settings.py syntax OK
  ✅ logger.py syntax OK
  ✅ api/app.py syntax OK
  ✅ config.settings importable
  ✅ src.utils.logger importable
  ✅ docs/BM25_FOUNDATION.md
  ✅ docs/bronze_schema.md
  ✅ LGPD_ALIGNMENT.md
  ✅ RESPONSIBLE_AI.md
  ✅ RANDOM_SEED = 42

📋 Academic Requirements:
  ✅ 21 Monitored Sources (20 Editorial + Reddit)
  ✅ PySpark · Distributed Computing · RDD MapReduce
  ✅ BM25 (rank-bm25) · TF-IDF vectorization
  ✅ Contextual Sentiment — PT-BR Financial Lexicon (NB05)
  ✅ Medallion Architecture: Bronze → Silver → Gold
  ✅ FastAPI REST · Streamlit Dashboard · Groq Chatbot (stubs)
  ✅ Responsible AI · LGPD · EU AI Act alignment
  ✅ CRISP-DM methodology (NB00–NB07 lifecycle)

🎉 NB00 COMPLETO — PRONTO PARA NB01
   Próximo: 01_data_in


## ✅ NB00 Completo

| Artefato | Consumido por |
|----------|---------------|
| `config/settings.py` | NB01–NB07 |
| `src/utils/logger.py` | NB01–NB07 |
| `api/app.py` | Fase 2 |
| `dashboard/Home.py` | Fase 2 |
| `dashboard/chatbot/groq_client.py` | Fase 2 |
| Docs de governança | Entrega acadêmica |

▶️ **Próximo:** `01_data_ingestion_bronze.ipynb`